[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/bsheese/cs377/blob/main/supplemental/Nested_CV_Pair.ipynb)

# Nested CV: How Good Is It, Honestly?
**CS 377 · companion to 17_2_1_4 Hyperparameter Tuning and 17_2_1_5 Nested CV · pair activity**

**Driver:** this notebook is yours. Your partner, the **Recorder**, has the **Nested CV quiz in Canvas** open, reads each question aloud and types your agreed answers.

**How the two fit together.** The quiz is the guide; this notebook is where you do the work for the quiz questions tagged **💻 NOTEBOOK**. Each section below is named after its quiz question (**Q3** here is **Q3** in the quiz). Questions tagged **💬 NO NOTEBOOK** have no section here: just talk them through.

**For each notebook question:** the code is written for you. The Recorder reads the question aloud; you find the matching section, make the small change marked **✏️** and run the cell (Shift+Enter). Read the result aloud and agree with your partner on what it means. Hints are under each section: try without them first, and open Hint 1 before Hint 2.

## Setup (run this first)
**Part 1 of the quiz (Q1-Q2) has no notebook work.** Run this cell while the Recorder reads Part 1; loading the data takes a few seconds.

It loads the Ames data and builds what every section uses:
- `X_train`, `X_test`, `y_train`, `y_test`: the same split as 17_2_1_4 (2,340 training houses, 585 test houses, 225 features, target = log sale price).
- `X`, `y`: all 2,925 houses, for nested CV (as in 17_2_1_5).
- `ALPHAS`: 20 Ridge alphas from 0.01 to 10,000 (17_2_1_4 used 100; 20 keeps every cell quick).
- `make_pipe(model)`: a `StandardScaler`, then the model, in a pipeline step named `'regressor'`.
- `dollars(y_true, y_pred)`: the mean absolute error in dollars, from log-price values.

In [ ]:
import warnings
warnings.filterwarnings('ignore')   # Lasso prints convergence warnings for small alphas

import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge, Lasso
from sklearn.model_selection import (GridSearchCV, KFold, cross_val_predict, cross_validate,
                                     train_test_split)
from sklearn.metrics import mean_absolute_error

# the Ames data, cleaned exactly as in 17_2_1_1 (same split as 17_2_1_4)
module_url = "https://raw.githubusercontent.com/bsheese/cs377/main/17_regression_crossval/17_2_1_MLR/ames_cleaning.py"
urllib.request.urlretrieve(module_url, "ames_cleaning.py")
from ames_cleaning import load_and_clean_ames

data_url = "https://raw.githubusercontent.com/bsheese/cs377/main/data/data_housing_ames.txt"
X_train, X_test, y_train, y_test = load_and_clean_ames(data_url)   # y is log sale price

# all 2,925 houses, for nested CV (as in 17_2_1_5)
X = pd.concat([X_train, X_test])
y = pd.concat([y_train, y_test])

ALPHAS = np.logspace(-2, 4, 20)          # 20 Ridge alphas from 0.01 to 10,000
LASSO_ALPHAS = np.logspace(-4, -1, 8)    # 8 Lasso alphas (extra credit Q7)


def make_pipe(model):
    """StandardScaler, then the model, in a step named 'regressor' (the same pipeline as 17_2_1_4)."""
    return Pipeline([('scaler', StandardScaler()), ('regressor', model)])


def dollars(y_true, y_pred):
    """Mean absolute error in dollars, from log-price values."""
    return mean_absolute_error(np.exp(y_true), np.exp(y_pred))


print("training houses:", len(X_train), "  test houses:", len(X_test), "  all houses:", len(X),
      "  features:", X.shape[1])
print("ALPHAS:", ", ".join(f"{a:.3g}" for a in ALPHAS))

# Part 2 (Q3-Q6): Tune, Then Measure Honestly

## Q3 · Tune once, the 17_2_1_4 way
> **Quiz Q3:** Run the Q3 cell. It fails. What does the error say, and why must the grid key be `'regressor__alpha'` and not `'alpha'`? Fix it and run again: which α wins, and what is the test MAE on `X_test`? If the pipeline step were named `'model'` instead of `'regressor'`, what else would have to change?

**✏️ Your change:** Change the one grid key the error points at, then run the cell again.

**Tell your Recorder:** what the error said and your fix, the winning α and the test MAE, then what a rename would change.

<details><summary>Hint 1</summary>

The error lists the parameters a Pipeline does have. `alpha` belongs to Ridge, which is the step named `'regressor'` inside `make_pipe`.

</details>

<details><summary>Hint 2</summary>

The key is the step name, two underscores, then the parameter: `'regressor__alpha'`.

</details>

In [ ]:
grid = GridSearchCV(make_pipe(Ridge()), {'alpha': ALPHAS},
                    cv=5, scoring='neg_mean_squared_error')
grid.fit(X_train, y_train)

print(f"winning alpha: {grid.best_params_['regressor__alpha']:.1f}")
print(f"test MAE:      ${dollars(y_test, grid.predict(X_test)):,.0f}")

## Q4 · Nested CV
> **Quiz Q4:** The Q4 cell runs the outer loop, but it hands `cross_val_predict` a plain Ridge pipeline (α = 1, no tuning). Run it and note the MAE. Then swap in the `inner` grid search, so every outer fold tunes its own α, and run again. What is the nested MAE, and how much did tuning buy? The cell also prints the α each outer fold chose. Why can the folds choose different α's, and what does it tell you that they are so close? (Compare them with `ALPHAS`, printed by Setup.)

**✏️ Your change:** In the `cross_val_predict(...)` line, replace `make_pipe(Ridge())` with `inner`, then run the cell again.

**Tell your Recorder:** the untuned and nested MAEs, the five alphas, then why they differ and what their closeness means.

<details><summary>Hint 1</summary>

`cross_val_predict` takes any estimator, including a whole `GridSearchCV`. Whatever you hand it is what gets fit in each outer fold.

</details>

<details><summary>Hint 2</summary>

Change `cross_val_predict(make_pipe(Ridge()), X, y, cv=5)` to `cross_val_predict(inner, X, y, cv=5)`.

</details>

In [ ]:
inner = GridSearchCV(make_pipe(Ridge()), {'regressor__alpha': ALPHAS},
                     cv=5, scoring='neg_mean_squared_error')

# the OUTER loop: 5 folds over all 2,925 houses
nested_preds = cross_val_predict(make_pipe(Ridge()), X, y, cv=5)   # ✏️ swap in the inner loop
print(f"MAE: ${dollars(y, nested_preds):,.0f}")

# the alpha that each outer fold's inner loop chose
folds = cross_validate(inner, X, y, cv=5, return_estimator=True)
print('alphas:', [round(float(g.best_params_['regressor__alpha']), 1) for g in folds['estimator']])

## Q5 · One test split, or ten?
> **Quiz Q5:** In Q3 the tuned model scored about $11,800 on `X_test`. In Q4 nested CV said about $12,750. A classmate says, "Nested CV made the model worse." Test it. The Q5 cell repeats the Q3 procedure (split, tune, score on the test set) for 10 different random splits. Replace `np.nan` with the test MAE in dollars, run it, and look at the plot. What range do the 10 test MAEs cover? Where does the Q3 split land? Where does the nested number land? What is wrong with the classmate's claim?

**✏️ Your change:** Replace `np.nan` with the test MAE in dollars, the same expression as in Q3 but with `X_te` and `y_te`.

**Tell your Recorder:** the range of the 10 MAEs, where Q3 and nested land, then your reply to the classmate.

<details><summary>Hint 1</summary>

In Q3 the test MAE was `dollars(y_test, grid.predict(X_test))`. Inside the loop the test set is `X_te`, `y_te`.

</details>

<details><summary>Hint 2</summary>

`mae = dollars(y_te, grid.predict(X_te))`

</details>

In [ ]:
grid = GridSearchCV(make_pipe(Ridge()), {'regressor__alpha': ALPHAS},
                    cv=5, scoring='neg_mean_squared_error')
maes = []
for seed in range(10):
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=seed)
    grid.fit(X_tr, y_tr)                  # tune alpha on this split's training houses
    mae = np.nan                          # ✏️ replace np.nan: this split's test MAE in dollars
    maes.append(mae)
    print(f"split {seed}: alpha {grid.best_params_['regressor__alpha']:7.1f}   test MAE ${mae:,.0f}")

plt.figure(figsize=(8, 2.2))
plt.plot(maes, [0] * len(maes), 'o', ms=10, alpha=0.6, label='10 random test splits')
plt.axvline(11763, color='tab:red', label='Q3: the 17_2_1_4 split')
plt.axvline(12747, color='tab:green', label='Q4: nested CV')
plt.yticks([]); plt.xlabel('test MAE ($)'); plt.legend(loc='upper left', bbox_to_anchor=(1, 1))
plt.show()

## Q6 · Ship it
> **Quiz Q6:** Nested CV built five tuned models, one per outer fold. Which of them do you deploy? The Q6 cell builds the production model, but it fits on `X_train` only. Change it to use all the data and run it. Which α does it choose? Your manager asks, "About how far off will this model's price predictions be?" Which number do you give, and why that one: the Q3 test MAE, the nested MAE, or something else?

**✏️ Your change:** Change `fit_X, fit_y = X_train, y_train` to use all the data.

**Tell your Recorder:** which model you deploy, the production alpha, then the number you give the manager and why.

<details><summary>Hint 1</summary>

`X_train`, `y_train` hold 2,340 houses. `X`, `y` hold all 2,925.

</details>

<details><summary>Hint 2</summary>

`fit_X, fit_y = X, y`

</details>

In [ ]:
production = GridSearchCV(make_pipe(Ridge()), {'regressor__alpha': ALPHAS},
                          cv=5, scoring='neg_mean_squared_error')
fit_X, fit_y = X_train, y_train            # ✏️ fit on all 2,925 houses instead
production.fit(fit_X, fit_y)

print(f"production alpha: {production.best_params_['regressor__alpha']:.1f}")
print(f"houses used: {len(fit_y)}")

# Extra credit (Q7-Q8): Settle It
Only if you have finished Q1-Q6.

## ⭐ Q7 · Ridge vs. Lasso, honestly
> **Quiz Q7:** Ridge vs. Lasso ended with an untuned comparison that no pair reached. Settle it. The Q7 cell runs the Q4 nested CV. Change it to Lasso: use `Lasso(max_iter=10000)` and `LASSO_ALPHAS`. What nested MAE does Lasso get, compared with Ridge's? Which α's did the folds choose, and how many features does each fold's model keep? Which would you recommend, and why?

**✏️ Your change:** In the first line, change `Ridge()` to `Lasso(max_iter=10000)` and `ALPHAS` to `LASSO_ALPHAS`, then run. (Lasso is slower than Ridge.)

**Tell your Recorder:** Lasso's nested MAE next to Ridge's, the alphas and feature counts, then your recommendation.

<details><summary>Hint 1</summary>

Only the first line changes. The rest of the cell works for either model.

</details>

<details><summary>Hint 2</summary>

`model, grid_values = Lasso(max_iter=10000), LASSO_ALPHAS`

</details>

In [ ]:
model, grid_values = Ridge(), ALPHAS          # ✏️ Lasso(max_iter=10000), LASSO_ALPHAS

inner = GridSearchCV(make_pipe(model), {'regressor__alpha': grid_values},
                     cv=5, scoring='neg_mean_squared_error')
folds = cross_validate(inner, X, y, cv=5, return_estimator=True, return_indices=True)

preds = pd.Series(np.nan, index=y.index)
for g, test_rows in zip(folds['estimator'], folds['indices']['test']):
    preds.iloc[test_rows] = g.predict(X.iloc[test_rows])
    kept = (g.best_estimator_.named_steps['regressor'].coef_ != 0).sum()
    print(f"alpha {g.best_params_['regressor__alpha']:.5f}   features kept: {kept}")
print(f"nested MAE: ${dollars(y, preds):,.0f}")

## ⭐ Q8 · Is the nested number steady?
> **Quiz Q8:** The Q8 cell runs nested CV five times. Run it: why are all five numbers identical? Now change the outer `cv=5` to `KFold(5, shuffle=True, random_state=s)` and run again (this cell is slow: it runs nested CV five times). How much do the five nested numbers move now, compared with the ten single splits in Q5? Why is the nested number so much steadier?

**✏️ Your change:** In the `cross_val_predict(...)` line, change `cv=5` to `cv=KFold(5, shuffle=True, random_state=s)`.

**Tell your Recorder:** why they were identical, the new range, then why nested is steadier.

<details><summary>Hint 1</summary>

`KFold` (imported in Setup) makes the folds. `shuffle=True` mixes the houses before cutting, and `random_state=s` gives a different mix on each run.

</details>

<details><summary>Hint 2</summary>

`cross_val_predict(inner, X, y, cv=KFold(5, shuffle=True, random_state=s))`

</details>

In [ ]:
inner = GridSearchCV(make_pipe(Ridge()), {'regressor__alpha': ALPHAS},
                     cv=5, scoring='neg_mean_squared_error')
for s in range(5):
    preds = cross_val_predict(inner, X, y, cv=5)     # ✏️ shuffle the outer folds with seed s
    print(f"run {s}: nested MAE ${dollars(y, preds):,.0f}")